# Methodology B - EfficientNet-B0 + VBLL (Variational Bayesian Last Layer)

Same strategy as Methodology A (MC Dropout), with one change: **the dense classification layer is replaced by a VBLL layer trained with an ELBO loss.**

- **Backbone:** EfficientNet-B0 (ImageNet pretrained), identical to Methodology A.
- **Head:** Variational Bayesian Last Layer - a full posterior distribution q(W) over the last-layer weights instead of one deterministic matrix.
- **Loss:** ELBO = expected classification log-likelihood (reparameterization trick) - KL(q(W) || prior) / N.
- **Uncertainty:** 30 posterior weight samples -> mean grade probabilities, final grade (argmax), confidence = (1 - std of top class) x 100, low-confidence flag (std > 0.15) -> **Refer for Manual Review**. Same interface as MC Dropout, so the two methodologies are directly comparable.
- **Explainability + fusion:** Grad-CAM (final conv block, COLORMAP_JET, 224x224, 60/40 blend) and the same lesion-evidence audit as Methodology A.
- **Fair A/B protocol:** same datasets, same preprocessing, same seed, same stratified splits as Methodology A.

Datasets: IDRiD (grades 0-4), APTOS 2019 (grades 0-4), ODIR healthy (capped, grade 0). Same paths as the Stage 1 notebook.

In [ ]:
# Kaggle setup
!pip -q install albumentations==1.4.24 scikit-learn opencv-python-headless

import os, glob, random, math, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
                             confusion_matrix, classification_report)

import albumentations as A

print('Torch:', torch.__version__)
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Dataset paths (same as the Stage 1 GANomaly notebook)
IDRID_ROOT = Path("/kaggle/input/datasets/lakshmiprathik/idrid-516/IDRiD")
ODIR_ROOT = Path("/kaggle/input/datasets/lakshmiprathik/odir-5k/ODIR-5K")
APTOS_ROOT = Path("/kaggle/input/datasets/mariaherrerot/aptos2019")
APTOS_CSV = APTOS_ROOT / "train_1.csv"
APTOS_IMAGE_DIR = (APTOS_ROOT / "train_images" / "train_images") if (APTOS_ROOT / "train_images" / "train_images").exists() else (APTOS_ROOT / "train_images")

WORK = Path("/kaggle/working/stage2_vbll")
CACHE = WORK / "cache"
CHECKPOINTS = WORK / "checkpoints"
for d in (WORK, CACHE, CHECKPOINTS):
    d.mkdir(parents=True, exist_ok=True)

# Config
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 45      # best epoch was ~23-25/25 -> schedule was still improving
LR_HEAD = 3e-4
LR_BACKBONE = 3e-5
PRIOR_SCALE = 1.0      # std of the Gaussian prior p(W) in the ELBO's KL term
TRAIN_W_SAMPLES = 10   # weight samples per training step (reparameterization trick)
PREDICT_PASSES = 30    # posterior weight samples at inference (same UX as MC Dropout)
TTA_VIEWS = 4       # flip views averaged at inference (test-time augmentation)
LOW_CONF_STD = 0.15    # top-class predictive std above this -> Refer for Manual Review
LESION_CAM_THR = 0.6   # CAM value counted as lesion evidence
LESION_HIGH = 0.08     # >8% retinal area strongly active contradicts a grade-0 call
LESION_LOW = 0.01      # <1% active contradicts a grade>=2 call
ODIR_CAP = 1000
SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

try:
    NUM_WORKERS = min(2, len(os.sched_getaffinity(0)))
except AttributeError:
    NUM_WORKERS = min(2, os.cpu_count() or 1)

GRADE_NAMES = {0: "No DR", 1: "Mild NPDR", 2: "Moderate NPDR", 3: "Severe NPDR", 4: "Proliferative DR"}

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything()

print("IDRiD root exists:", IDRID_ROOT.exists())
print("ODIR root exists:", ODIR_ROOT.exists())
print("APTOS root exists:", APTOS_ROOT.exists())
print("APTOS CSV exists:", APTOS_CSV.exists())
print("APTOS image folder exists:", APTOS_IMAGE_DIR.exists())
print("Device:", DEVICE, "| workers:", NUM_WORKERS)

## Dataset policy

Identical to Methodology A, so the A/B comparison is controlled:

- **IDRiD** and **APTOS** provide graded labels 0-4; **ODIR** contributes a capped sample of healthy grade-0 images (default 1000).
- Splits are **stratified by grade** (70 / 15 / 15, seed 42) with a leakage audit - the same seed and pipeline produce the same partitions as Methodology A.
- Class imbalance is handled with a `WeightedRandomSampler` plus augmentation. Headline metric: **quadratic weighted kappa (QWK)**, with accuracy and macro-F1 alongside.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def all_images(root):
    return [p for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]

def idrid_rows():
    rows = []
    for split in ["train", "validation", "test"]:
        for grade in range(5):
            folder = IDRID_ROOT / split / str(grade)
            if not folder.exists():
                print("Missing IDRiD folder:", folder)
                continue
            for image_path in all_images(folder):
                rows.append({"path": str(image_path), "grade": grade, "source": "idrid", "split": split})
    return rows

def aptos_rows():
    if not APTOS_CSV.exists():
        raise FileNotFoundError(f"APTOS CSV not found: {APTOS_CSV}")
    if not APTOS_IMAGE_DIR.exists():
        raise FileNotFoundError(f"APTOS image directory not found: {APTOS_IMAGE_DIR}")
    aptos_df = pd.read_csv(APTOS_CSV)
    print("APTOS columns:", list(aptos_df.columns), "| rows:", len(aptos_df))
    print(aptos_df["diagnosis"].value_counts().sort_index())
    rows = []
    for row in aptos_df.itertuples(index=False):
        image_id = str(row.id_code)
        candidates = [APTOS_IMAGE_DIR / f"{image_id}.png",
                      APTOS_IMAGE_DIR / f"{image_id}.jpg",
                      APTOS_IMAGE_DIR / f"{image_id}.jpeg"]
        image_path = next((p for p in candidates if p.exists()), None)
        if image_path is not None:
            rows.append({"path": str(image_path), "grade": int(row.diagnosis), "source": "aptos", "split": "all"})
    print("APTOS images matched:", len(rows))
    return rows

def odir_rows(cap=ODIR_CAP):
    image_paths = all_images(ODIR_ROOT)
    print("ODIR healthy images found:", len(image_paths))
    if len(image_paths) > cap:
        idx = np.random.RandomState(SEED).choice(len(image_paths), size=cap, replace=False)
        image_paths = [image_paths[i] for i in idx]
    rows = [{"path": str(p), "grade": 0, "source": "odir", "split": "all"} for p in image_paths]
    print("ODIR healthy images used:", len(rows))
    return rows

rows = idrid_rows() + aptos_rows() + odir_rows()
df = pd.DataFrame(rows)
if len(df) == 0:
    raise RuntimeError("No images were found. Check the dataset paths.")
df = df.drop_duplicates(subset="path").reset_index(drop=True)
df["grade"] = df["grade"].astype(int)

print("\nTotal unique images:", len(df))
display(pd.crosstab(df["source"], df["grade"]))

In [ ]:
def crop_square(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > 7
    if mask.sum() > 100:
        ys, xs = np.where(mask)
        img = img[ys.min():ys.max()+1, xs.min():xs.max()+1]
    h, w = img.shape[:2]
    s = max(h, w)
    canvas = np.zeros((s, s, 3), np.uint8)
    y = (s - h)//2; x = (s - w)//2
    canvas[y:y+h, x:x+w] = img
    return canvas

def preprocess(path, size=IMG_SIZE):
    img = cv2.imread(str(path))
    if img is None:
        return np.zeros((size, size, 3), np.uint8)
    img = crop_square(img)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

def save_cache(frame):
    cached = []
    for i, r in tqdm(frame.iterrows(), total=len(frame), desc="preprocessing"):
        out = CACHE / (str(i) + ".png")
        if not out.exists():
            cv2.imwrite(str(out), preprocess(r.path))
        cached.append(str(out))
    frame = frame.copy()
    frame["cache_path"] = cached
    return frame

df = save_cache(df)

In [ ]:
# 70 / 15 / 15 stratified by grade (same seed as Methodology A -> same partitions)
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["grade"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["grade"])

splits = {
    "train": train_df.reset_index(drop=True),
    "val": val_df.reset_index(drop=True),
    "test": test_df.reset_index(drop=True),
}
for k, v in splits.items():
    print(k, len(v), dict(v["grade"].value_counts().sort_index()))

assert not (set(splits["train"].cache_path) & set(splits["val"].cache_path))
assert not (set(splits["train"].cache_path) & set(splits["test"].cache_path))
assert not (set(splits["val"].cache_path) & set(splits["test"].cache_path))
print("Leakage audit: PASSED")

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], np.float32)

train_tf = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=30, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
])

class SeverityDataset(Dataset):
    def __init__(self, frame, train=False):
        self.frame = frame.reset_index(drop=True)
        self.train = train
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        r = self.frame.iloc[i]
        img = cv2.cvtColor(cv2.imread(r.cache_path), cv2.COLOR_BGR2RGB)
        if self.train:
            img = train_tf(image=img)["image"]
        x = (img.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
        return torch.from_numpy(x).permute(2, 0, 1), int(r.grade)

counts = splits["train"]["grade"].value_counts().sort_index().values.astype(np.float64)
class_weights = 1.0 / np.maximum(counts, 1)
sample_weights = splits["train"]["grade"].map(lambda g: class_weights[int(g)]).values
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(SeverityDataset(splits["train"], True), batch_size=BATCH_SIZE,
                          sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(SeverityDataset(splits["val"]), batch_size=BATCH_SIZE * 2,
                        num_workers=NUM_WORKERS)
test_loader = DataLoader(SeverityDataset(splits["test"]), batch_size=BATCH_SIZE * 2,
                         num_workers=NUM_WORKERS)

print("Train class counts:", counts.astype(int).tolist())
print("Sampler class weights:", np.round(class_weights, 5).tolist())

## Model: EfficientNet-B0 + VBLL

The backbone is identical to Methodology A, but the classification head is a **Variational Bayesian Last Layer**. Instead of one deterministic weight matrix, the head maintains a *posterior distribution* over its weights, q(W) = N(mu, sigma^2) (mean-field / diagonal Gaussian), and is trained with the **ELBO loss**:

`ELBO = E_q[ log p(y | x, W) ] - KL( q(W) || p(W) ) / N`

- **First term:** classification fit, averaged over weight samples via the reparameterization trick (`TRAIN_W_SAMPLES` per step).
- **Second term:** pulls the posterior toward the Gaussian prior p(W) = N(0, `PRIOR_SCALE`^2 * I), scaled by 1/N (N = training set size) so it acts as principled Bayesian regularisation rather than a hand-tuned weight decay.

At inference, weight samples from q(W) play the same role as MC Dropout passes in Methodology A - but the spread is **learned**, not injected, so the uncertainty is typically better calibrated. Deterministic predictions (for metrics and Grad-CAM) use the posterior-mean forward pass.

In [ ]:
class VBLLClassifier(nn.Module):
    """Variational Bayesian Last Layer (mean-field Gaussian) trained with ELBO."""
    def __init__(self, in_features, out_features,
                 prior_scale=PRIOR_SCALE, train_samples=TRAIN_W_SAMPLES):
        super().__init__()
        self.prior_scale = prior_scale
        self.train_samples = train_samples
        # Variational posterior q(W) = N(w_mu, diag(exp(2 * w_log_sigma)))
        self.w_mu = nn.Parameter(torch.randn(out_features, in_features) * 0.05)
        self.w_log_sigma = nn.Parameter(torch.full((out_features, in_features), -5.0))
        self.b_mu = nn.Parameter(torch.zeros(out_features))
        self.b_log_sigma = nn.Parameter(torch.full((out_features,), -5.0))

    def forward(self, f):
        # Deterministic posterior-mean logits (metrics, Grad-CAM).
        return F.linear(f, self.w_mu, self.b_mu)

    def _kl(self):
        # KL( N(mu, sigma^2) || N(0, prior_scale^2) ), analytic, diagonal case.
        def kl(mu, log_sigma):
            var = (2.0 * log_sigma).exp()
            return 0.5 * ((mu * mu + var) / (self.prior_scale ** 2) - 1.0
                          + 2.0 * math.log(self.prior_scale) - 2.0 * log_sigma).sum()
        return kl(self.w_mu, self.w_log_sigma) + kl(self.b_mu, self.b_log_sigma)

    def elbo_loss(self, f, y, n_data):
        # E_q[log p(y|x,W)] via reparameterization trick, minus KL(q||prior)/N.
        logps = []
        for _ in range(self.train_samples):
            w = self.w_mu + self.w_log_sigma.exp() * torch.randn_like(self.w_mu)
            b = self.b_mu + self.b_log_sigma.exp() * torch.randn_like(self.b_mu)
            logp = F.log_softmax(F.linear(f, w, b), dim=1)
            logps.append(logp.gather(1, y.unsqueeze(1)).mean())
        loglik = torch.stack(logps).mean()
        elbo = loglik - self._kl() / n_data
        return -elbo   # minimise the negative ELBO

    @torch.no_grad()
    def sample_probs(self, f, passes):
        # Posterior predictive: softmax under `passes` weight samples from q(W).
        probs = []
        for _ in range(passes):
            w = self.w_mu + self.w_log_sigma.exp() * torch.randn_like(self.w_mu)
            b = self.b_mu + self.b_log_sigma.exp() * torch.randn_like(self.b_mu)
            probs.append(F.softmax(F.linear(f, w, b), dim=1))
        return torch.stack(probs)   # (passes, batch, out_features)

class VBLLNet(nn.Module):
    """EfficientNet-B0 feature extractor + VBLL head."""
    def __init__(self, backbone):
        super().__init__()
        self.features = backbone.features          # Grad-CAM targets features[-1]
        self.avgpool = backbone.avgpool
        in_features = backbone.classifier[1].in_features
        self.vbll = VBLLClassifier(in_features, 5)

    def extract_features(self, x):
        f = self.features(x)
        f = self.avgpool(f)
        return torch.flatten(f, 1)

    def forward(self, x):
        return self.vbll(self.extract_features(x))          # posterior-mean logits

    def elbo_loss(self, x, y, n_data):
        return self.vbll.elbo_loss(self.extract_features(x), y, n_data)

    def sample_probs(self, x, passes):
        return self.vbll.sample_probs(self.extract_features(x), passes)

model = VBLLNet(models.efficientnet_b0(weights="IMAGENET1K_V1")).to(DEVICE)
print("Total params:", sum(p.numel() for p in model.parameters()))
print("Bayesian head params (mu + log_sigma):", sum(p.numel() for p in model.vbll.parameters()))

In [ ]:
def evaluate(m, loader):
    m.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            logits = m(x.to(DEVICE))          # posterior-mean logits
            ys.append(y.numpy())
            ps.append(logits.argmax(1).cpu().numpy())
    y = np.concatenate(ys); p = np.concatenate(ps)
    return {"qwk": cohen_kappa_score(y, p, weights="quadratic"),
            "acc": accuracy_score(y, p),
            "f1": f1_score(y, p, average="macro", zero_division=0)}

def train_model():
    n_data = float(len(splits["train"]))      # scales the KL term in the ELBO
    head_params = list(model.vbll.parameters())
    backbone_params = [p for n, p in model.named_parameters() if not n.startswith("vbll")]
    opt = torch.optim.AdamW([
        {"params": backbone_params, "lr": LR_BACKBONE},
        {"params": head_params, "lr": LR_HEAD},
    ], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    AMP_ENABLED = DEVICE == "cuda" or getattr(DEVICE, "type", None) == "cuda"
    if hasattr(torch, "amp"):
        try:
            scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
        except TypeError:
            scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)
    else:
        scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

    def autocast_context():
        if hasattr(torch, "amp"):
            return torch.amp.autocast(device_type="cuda", enabled=AMP_ENABLED)
        return torch.cuda.amp.autocast(enabled=AMP_ENABLED)

    best_qwk = -1.0
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, total_n = 0.0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
        for x, y in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast_context():
                loss = model.elbo_loss(x, y, n_data)     # <-- ELBO loss, not cross-entropy
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            total_loss += loss.item() * x.size(0)
            total_n += x.size(0)
            pbar.set_postfix({"elbo": f"{loss.item():.3f}"})
        sched.step()

        val = evaluate(model, val_loader)
        history.append({"epoch": epoch, "loss": total_loss / max(1, total_n),
                        **{f"val_{k}": v for k, v in val.items()}})
        if val["qwk"] > best_qwk:
            best_qwk = val["qwk"]
            torch.save({"model": model.state_dict(), "epoch": epoch, **val},
                       CHECKPOINTS / "stage2_vbll_best.pt")
        print(f"Epoch {epoch:03d}/{EPOCHS} | ELBO loss={total_loss / max(1, total_n):.4f} | "
              f"val QWK={val['qwk']:.4f} | val acc={val['acc']:.4f} | val F1={val['f1']:.4f}")

    pd.DataFrame(history).to_csv(WORK / "history.csv", index=False)
    ckpt = torch.load(CHECKPOINTS / "stage2_vbll_best.pt", map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"])
    model.eval()
    print(f"\nBest checkpoint loaded from epoch {ckpt['epoch']} (val QWK={ckpt['qwk']:.4f})")

train_model()

In [ ]:
# Held-out test evaluation - standard and with TTA (test-time augmentation)
# TTA averages softmax probabilities over flips: free accuracy, no retraining.

def tta_views(x):
    # original, horizontal flip, vertical flip, both.
    # Fundus images have no left/right orientation semantics, so flips are safe.
    return [x, torch.flip(x, dims=[3]), torch.flip(x, dims=[2]), torch.flip(x, dims=[2, 3])][:TTA_VIEWS]

@torch.no_grad()
def eval_predictions(loader, use_tta):
    model.eval()
    ys, ps = [], []
    for x, y in tqdm(loader, desc="test+TTA" if use_tta else "test"):
        x = x.to(DEVICE)
        if use_tta:
            xb = torch.cat(tta_views(x), 0)                      # (views*B, 3, H, W)
            p = F.softmax(model(xb), 1)
            p = p.view(len(tta_views(x)), x.size(0), 5).mean(0)  # average over views
        else:
            p = F.softmax(model(x), 1)
        ys.append(y.numpy())
        ps.append(p.argmax(1).cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)

def report(y_true, y_pred, tag):
    print(f"--- {tag} ---")
    print("accuracy:         ", accuracy_score(y_true, y_pred))
    print("within-one-grade: ", float((np.abs(y_true - y_pred) <= 1).mean()))
    print("macro F1:         ", f1_score(y_true, y_pred, average="macro", zero_division=0))
    print("QWK:              ", cohen_kappa_score(y_true, y_pred, weights="quadratic"))
    print()

y_true, y_pred = eval_predictions(test_loader, use_tta=False)
report(y_true, y_pred, "standard (posterior mean)")
y_true, y_pred = eval_predictions(test_loader, use_tta=True)
report(y_true, y_pred, "with TTA (headline numbers)")

print(classification_report(y_true, y_pred, labels=list(range(5)),
                            target_names=[GRADE_NAMES[i] for i in range(5)], zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4])
recall = cm.diagonal() / np.maximum(cm.sum(1), 1)
for i in range(5):
    print(f"  grade {i} ({GRADE_NAMES[i]}): recall {recall[i]:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels([GRADE_NAMES[i] for i in range(5)], rotation=45, ha="right")
ax.set_yticklabels([GRADE_NAMES[i] for i in range(5)])
for i in range(5):
    for j in range(5):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="black" if cm[i, j] < cm.max() / 2 else "white")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Held-out test confusion matrix (with TTA)")
plt.colorbar(im); plt.tight_layout(); plt.show()

## Inference: posterior sampling + Grad-CAM + hybrid fusion

For a single image the pipeline computes:

1. **Posterior sampling (30 passes):** sample W ~ q(W), softmax each pass -> mean probabilities `p`, per-class std.
   - final grade = `argmax(p)`
   - confidence = `(1 - std[predicted class]) * 100`
   - low-confidence flag: `std > 0.15` -> Refer for Manual Review
2. **Grad-CAM** on `model.features[-1]` (final conv block), gradients through the posterior-mean forward: ReLU-weighted activation map, normalised, `COLORMAP_JET`, resized to 224x224, blended 60% original / 40% heatmap.
3. **Hybrid lesion-evidence + severity fusion:** identical to Methodology A - lesion load `L` (fraction of retina with CAM >= 0.6) audits the predicted grade; contradictions or low confidence -> **Refer for Manual Review**.

The uncertainty interface is deliberately identical to Methodology A, so you can compare *which model's uncertainty catches errors better* - that is the real A/B question.

**TTA:** all reported probabilities (test metrics and single-image inference) are averaged over 4 flip views (original + horizontal + vertical + both) - free accuracy, no retraining.

In [ ]:
# ---------------- Posterior predictive (VBLL) + TTA ----------------
@torch.no_grad()
def vbll_predict(x, passes=PREDICT_PASSES):
    model.eval()
    xb = torch.cat(tta_views(x), 0)              # TTA views stacked in the batch dim
    probs = model.sample_probs(xb, passes).cpu().numpy()  # (passes, views, 5)
    probs = probs.reshape(-1, probs.shape[-1])            # (passes * views, 5)
    mean_p = probs.mean(0)
    std_p = probs.std(0)
    grade = int(mean_p.argmax())
    top_std = float(std_p[grade])
    return {"mean_probs": mean_p, "std_probs": std_p, "grade": grade,
            "top_std": top_std, "confidence": (1.0 - top_std) * 100.0,
            "low_confidence": top_std > LOW_CONF_STD}

# ---------------- Grad-CAM ----------------
class GradCAM:
    def __init__(self, m, target_layer):
        self.m = m
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._fwd)
        target_layer.register_full_backward_hook(self._bwd)
    def _fwd(self, module, inp, out):
        self.activations = out.detach()
    def _bwd(self, module, gin, gout):
        self.gradients = gout[0].detach()
    def generate(self, x, class_idx):
        self.m.zero_grad(set_to_none=True)
        self.m(x)[0, class_idx].backward()
        w = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.activations).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
        cam = cam[0, 0].cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

gradcam = GradCAM(model, model.features[-1])   # final conv block of EfficientNet-B0

# ---------------- Hybrid lesion-evidence + severity fusion ----------------
def lesion_load(cam, img_rgb):
    mask = img_rgb.mean(2) > 7                  # retinal area, ignore black border
    if mask.sum() < 100:
        mask = np.ones(cam.shape, bool)
    return float((cam[mask] >= LESION_CAM_THR).mean())

def hybrid_decision(mc, cam, img_rgb):
    p = mc["mean_probs"]
    grade = int(p.argmax())
    severity_index = float(np.dot(np.arange(5), p))
    L = lesion_load(cam, img_rgb)
    flags = []
    if L > LESION_HIGH and grade == 0:
        flags.append(f"lesion evidence ({L:.1%} of retina) contradicts grade 0")
    if L < LESION_LOW and grade >= 2:
        flags.append(f"weak lesion evidence ({L:.1%}) for grade >= 2")
    if mc["low_confidence"]:
        flags.append(f"posterior std {mc['top_std']:.3f} > {LOW_CONF_STD}")
    return {"grade": grade, "severity_index": severity_index, "lesion_load": L,
            "flags": flags,
            "action": ("Refer for Manual Review" if flags
                       else f"Proceed - grade {grade} ({GRADE_NAMES[grade]})")}

In [ ]:
def load_tensor(path):
    # Same pipeline as the training cache: crop + CLAHE + resize 224, then ImageNet normalisation.
    img = cv2.cvtColor(preprocess(path), cv2.COLOR_BGR2RGB)
    x = (img.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
    x = torch.from_numpy(x).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
    return img, x

def analyze_image(path, show=True):
    img_rgb, x = load_tensor(path)
    mc = vbll_predict(x)
    model.eval()                                   # deterministic posterior-mean CAM
    cam = gradcam.generate(x, mc["grade"])
    fusion = hybrid_decision(mc, cam, img_rgb)

    if show:
        heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
        heat = cv2.cvtColor(cv2.resize(heat, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_BGR2RGB)
        blend = cv2.addWeighted(img_rgb, 0.6, heat, 0.4, 0)

        fig = plt.figure(figsize=(16, 5))
        gs = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1, 1.3])
        for k, (im, t) in enumerate([(img_rgb, "Original"),
                                     (heat, "Grad-CAM (COLORMAP_JET)"),
                                     (blend, "Blend 60% original / 40% heatmap")]):
            ax = fig.add_subplot(gs[0, k])
            ax.imshow(im); ax.set_title(t); ax.axis("off")
        ax = fig.add_subplot(gs[0, 3])
        ax.bar(np.arange(5), mc["mean_probs"], yerr=mc["std_probs"], capsize=4, color="darkorange")
        ax.set_xticks(np.arange(5))
        ax.set_xlabel("grade"); ax.set_ylabel("mean prob (+/- posterior std)")
        ax.set_title("VBLL posterior + TTA (30 samples x 4 views)")
        plt.tight_layout(); plt.show()

    print("=" * 58)
    print("Image:           ", path)
    print("Predicted grade: ", mc["grade"], "-", GRADE_NAMES[mc["grade"]])
    print("Severity index:  ", f"{fusion['severity_index']:.2f} / 4 (continuous)")
    print("Confidence:      ", f"{mc['confidence']:.1f}%  (top-class posterior std = {mc['top_std']:.3f})")
    print("Lesion load:     ", f"{fusion['lesion_load']:.1%} of retinal area (CAM >= {LESION_CAM_THR})")
    print("Probabilities:   ", {GRADE_NAMES[i]: round(float(mc['mean_probs'][i]), 3) for i in range(5)})
    if fusion["flags"]:
        print("Audit flags:")
        for fl in fusion["flags"]:
            print("  -", fl)
    print("FINAL ACTION:    ", fusion["action"])
    print("=" * 58)
    return {**mc, **fusion}

In [ ]:
# --- Single random held-out test image ---
demo = splits["test"].sample(1).iloc[0]
print("True grade:", int(demo.grade), "-", GRADE_NAMES[int(demo.grade)])
analyze_image(demo.path)

# --- Sanity table: 12 random test images, never judge from one image ---
sample = splits["test"].sample(12, random_state=SEED).reset_index(drop=True)
rows = []
for r in tqdm(sample.itertuples(index=False), total=len(sample), desc="sanity"):
    _, x = load_tensor(r.path)
    mc = vbll_predict(x)
    rows.append({"image": Path(r.path).name, "true": int(r.grade), "pred": mc["grade"],
                 "conf%": round(mc["confidence"], 1), "post_std": round(mc["top_std"], 3),
                 "low_conf": mc["low_confidence"]})
sanity = pd.DataFrame(rows)
display(sanity)
print(f"Accuracy on these 12: {(sanity['true'] == sanity['pred']).mean():.2%} "
      "(small sample - trust the full test metrics, not this)")

In [ ]:
# ============================================================
# TEST ANY IMAGE - paste any image path below and run this cell
# ============================================================
img_path = "/kaggle/input/datasets/lakshmiprathik/idrid-516/IDRiD/test/4/IDRiD_001.jpg"  # <-- change this to your image

if not Path(img_path).exists():
    print("File not found:", img_path)
    print("Tip: open the Kaggle file browser (right panel), navigate to your image,")
    print("     copy its path, and paste it above. Example sources:")
    print("     /kaggle/input/...  (any attached dataset)")
    print("     /kaggle/working/... (files saved by this or the Stage 1 notebook)")
else:
    analyze_image(img_path)   # VBLL posterior sampling + Grad-CAM + hybrid fusion report


In [ ]:
# Save artifacts for deployment / thesis
config = {
    "methodology": "B - EfficientNet-B0 + VBLL (Variational Bayesian Last Layer), ELBO loss",
    "img_size": IMG_SIZE,
    "backbone": "efficientnet_b0 (IMAGENET1K_V1)",
    "head": "VBLLClassifier, mean-field Gaussian posterior",
    "prior_scale": PRIOR_SCALE,
    "train_w_samples": TRAIN_W_SAMPLES,
    "predict_passes": PREDICT_PASSES,
    "tta_views": TTA_VIEWS,
    "low_conf_std": LOW_CONF_STD,
    "lesion_cam_thr": LESION_CAM_THR,
    "lesion_high": LESION_HIGH,
    "lesion_low": LESION_LOW,
    "grade_names": GRADE_NAMES,
    "preprocessing": "crop + CLAHE + resize 224 (same as Stage 1 and Methodology A)",
    "fusion": "argmax grade + Grad-CAM lesion-evidence audit + posterior-std confidence audit",
    "integration": "consume images flagged by the Stage 1 GANomaly gate (anomaly_score > threshold)",
}
with open(WORK / "stage2_vbll_config.json", "w") as f:
    json.dump(config, f, indent=2)
pd.DataFrame({"y_true": y_true, "y_pred": y_pred}).to_csv(WORK / "test_predictions.csv", index=False)
print("Checkpoint:", CHECKPOINTS / "stage2_vbll_best.pt")
print("Config:    ", WORK / "stage2_vbll_config.json")
print("Predictions:", WORK / "test_predictions.csv")

## Caveats

- Same data, splits, seed and preprocessing as Methodology A -> the A/B comparison is controlled. Compare QWK / accuracy **and** the usefulness of the uncertainty (does the low-confidence flag catch the misclassified images in the sanity table?).
- The VBLL here uses a **mean-field (diagonal) Gaussian posterior** - cheap and stable. A full-covariance posterior captures weight correlations at much higher memory cost; not needed at this scale.
- The KL term is scaled by 1/N; if the model underfits or overfits, tune `PRIOR_SCALE` and `TRAIN_W_SAMPLES` in the config cell.
- Posterior stds from a learned posterior can differ in scale from MC-Dropout stds - if the Refer flag fires too rarely or too often, recalibrate `LOW_CONF_STD` on the validation split, not the test split.
- External validation on an untouched clinic dataset is still required before any clinical claim.